<a href="https://colab.research.google.com/github/izm4824/DCF/blob/main/DCF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import yfinance as yf
import pandas as pd
import numpy as np

#Fetch NVDA data from the cloud
print("Fetching NVDA data")
nvda = yf.Ticker("NVDA")

#Extract the annual income using Pandas
df_income = nvda.financials.T

#Pull out 'Total revenue' column into a clean Pandas DataFrame
df_analysis = pd.DataFrame(df_income['Total Revenue'])

#Calculate Year-over-Year Revenue Growth Rate
df_analysis['YoY Growth Rate'] = df_analysis['Total Revenue'].pct_change(periods=-1)

#Clean up the visual formatting so it prints in Billions($B) and Percentages(%)
pd.set_option('display.float_format',lambda x:f'${x/1e9:.2f}B' if x > 1e9 else f'{x:.2%}')

print("\n NVIDIA HISTORICAL REVENUE CORE ANALYSIS ")
print(df_analysis)


Fetching NVDA data

 NVIDIA HISTORICAL REVENUE CORE ANALYSIS 
            Total Revenue  YoY Growth Rate
2026-01-31       $215.94B           65.47%
2025-01-31       $130.50B          114.20%
2024-01-31        $60.92B          125.85%
2023-01-31        $26.97B            0.00%
2022-01-31            NaN              NaN


/tmp/ipykernel_1385/3777680434.py:16: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_analysis['YoY Growth Rate'] = df_analysis['Total Revenue'].pct_change(periods=-1)


In [13]:
# 1. Pull the Cash Flow Statement
df_cashflow = nvda.cashflow.T

# 2. Extract Operating Cash Flow and Capital Expenditures (CapEx)
df_fcf = pd.DataFrame({
    'Operating Cash Flow': df_cashflow['Cash Flow From Continuing Operating Activities'],
    'CapEx': df_cashflow['Capital Expenditure']
})

# 3. Calculate Free Cash Flow to Firm (FCFF)
df_fcf['Free Cash Flow (FCFF)'] = df_fcf['Operating Cash Flow'] - df_fcf['CapEx'].abs()

print("NVIDIA HISTORICAL FREE CASH FLOW BASELINE")
print(df_fcf)

NVIDIA HISTORICAL FREE CASH FLOW BASELINE
            Operating Cash Flow             CapEx  Free Cash Flow (FCFF)
2026-01-31             $102.72B -604200000000.00%                $96.68B
2025-01-31              $64.09B -323600000000.00%                $60.85B
2024-01-31              $28.09B -106900000000.00%                $27.02B
2023-01-31               $5.64B -183300000000.00%                 $3.81B
2022-01-31                  NaN               NaN                    NaN


In [14]:
# --- THE FINAL DCF INTRINSIC VALUATION SYSTEM ---

# 1. Define our starting baseline (From your exact Colab output)
base_fcf = 96.68e9           # $96.68 Billion from 2026
shares_outstanding = 24.2e9  # Roughly 24.2 Billion shares of NVDA in circulation

# 2. Define our Macroeconomic & Valuation Assumptions
wacc = 0.095                 # 9.5% Cost of Capital (Discount Rate)
terminal_growth = 0.03       # 3.0% Perpetual Growth Rate (matching long-term global GDP)
forecast_years = [1, 2, 3, 4, 5]

# 3. Forecast future FCF with a decelerating growth rate (Econ logic)
# Year 1 starts at 25% growth, dropping down to 5% by Year 5
growth_rates = [0.25, 0.20, 0.15, 0.10, 0.05]

projected_cash_flows = []
current_fcf = base_fcf
for g in growth_rates:
    current_fcf *= (1 + g)
    projected_cash_flows.append(current_fcf)

# 4. Create a clean matrix to calculate Present Values
df_dcf = pd.DataFrame({
    'Year': forecast_years,
    'Growth Rate': growth_rates,
    'Forecasted FCF': projected_cash_flows
})

# Calculate the Discount Factors: (1 + WACC)^t
df_dcf['Discount Factor'] = (1 + wacc) ** df_dcf['Year']

# Calculate the Present Value (PV) of each cash flow row
df_dcf['Present Value of FCF'] = df_dcf['Forecasted FCF'] / df_dcf['Discount Factor']

# 5. Calculate the Terminal Value (Value of all cash beyond Year 5)
final_year_fcf = projected_cash_flows[-1]
terminal_value = (final_year_fcf * (1 + terminal_growth)) / (wacc - terminal_growth)
pv_of_terminal_value = terminal_value / ((1 + wacc) ** 5)

# 6. Aggregate everything to find the Intrinsic Share Price
sum_pv_stage1 = df_dcf['Present Value of FCF'].sum()
total_enterprise_value = sum_pv_stage1 + pv_of_terminal_value
intrinsic_value_per_share = total_enterprise_value / shares_outstanding

# Reset formatting to display
pd.set_option('display.float_format', lambda x: f'${x/1e9:.2f}B' if x > 1e9 else (f'{x:.1%}' if x < 1 else f'${x:.2f}'))

print("5-YEAR TIME-VALUE DISCOUNTING PIPELINE")
print(df_dcf[['Year', 'Growth Rate', 'Forecasted FCF', 'Present Value of FCF']])

print("\nFINAL ECONOMIC INTRINSIC VALUATION SUMMARY")
print(f"PV of 5-Year Explicit Cash Flows:        ${sum_pv_stage1/1e9:.2f}B")
print(f"PV of Stable Steady-State Terminal Value: ${pv_of_terminal_value/1e9:.2f}B")
print(f"Total Implied Enterprise Value of Firm:   ${total_enterprise_value/1e9:.2f}B")
print(f"INTRINSIC VALUE PER SHARE:             ${intrinsic_value_per_share:.2f}")


5-YEAR TIME-VALUE DISCOUNTING PIPELINE
   Year  Growth Rate  Forecasted FCF  Present Value of FCF
0     1        25.0%        $120.85B              $110.37B
1     2        20.0%        $145.02B              $120.95B
2     3        15.0%        $166.77B              $127.02B
3     4        10.0%        $183.45B              $127.60B
4     5         5.0%        $192.62B              $122.36B

FINAL ECONOMIC INTRINSIC VALUATION SUMMARY
PV of 5-Year Explicit Cash Flows:        $608.30B
PV of Stable Steady-State Terminal Value: $1938.92B
Total Implied Enterprise Value of Firm:   $2547.22B
INTRINSIC VALUE PER SHARE:             $105.26
